In [199]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load Database

In [200]:
from pathlib import Path
DB_PATH = Path.cwd().parent / "database" / "car_sales.parquet"

# Read database
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
df = pd.read_parquet(DB_PATH)
df.head()

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47


In [201]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23905 entries, 0 to 23904
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   car_id           23905 non-null  object 
 1   date             23905 non-null  object 
 2   day              23905 non-null  int32  
 3   month            23905 non-null  int32  
 4   year             23905 non-null  int32  
 5   customer_name    23905 non-null  object 
 6   gender           23905 non-null  object 
 7   dealer_name      23905 non-null  object 
 8   company          23905 non-null  object 
 9   model            23905 non-null  object 
 10  engine           23905 non-null  object 
 11  transmission     23905 non-null  object 
 12  color            23905 non-null  object 
 13  dealer_no_       23905 non-null  object 
 14  body_style       23905 non-null  object 
 15  phone            23905 non-null  int64  
 16  dealer_region    23905 non-null  object 
 17  price       

## Feature engineering

In [202]:
# Create column feature day_of_week, is_weekend, is_workday
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['is_workday'] = df['day_of_week'].apply(lambda x: 1 if x < 5 else 0)

In [203]:
# Build Season feature to define the season of the year based on the month
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"
    
df["season"] = df["month"].apply(season)

In [204]:
# Adjust date into dividen specific periods
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

In [205]:
# Build feature price_band as arrangement of price into 4 bands
df["price_band"] = pd.qcut(
    df["price"], q=5, labels=[
        "Budget","Economy", "Mid", "Premium", "Luxury"
    ]
)

In [206]:
# Create a feature of discount_level
df['discount_level'] = pd.cut(
    df['discount'], bins=[0,0.05,0.1,0.15,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Extreme"
    ]
)

# Create feature of weekend only discount by multiplying is_weekend and discount
df['weekend_discount'] = (df['is_weekend'] * df['discount'])

In [207]:
# Aggregate for revenue featured
daily_sales = (
    df.groupby('date')
        .agg(
            revenue=('sales', 'sum'),
            quantity=('quantity', 'sum'),
            avg_discount=('discount', 'mean'),
            avg_price=('price', 'mean'),
            customers=('customer_name', 'nunique')
        )
        .reset_index()
)

# Transform to revenue
revenue = daily_sales['revenue']

In [208]:
# Create lag features of daily_sales

# Create lg
daily_sales["lag_1"] = daily_sales["revenue"].shift(1)

# Create last week lag
daily_sales["lag_7"] = daily_sales["revenue"].shift(7)

# Create last month lag
daily_sales["lag_30"] = daily_sales["revenue"].shift(30)

In [209]:
# Create features of rolling mean and rolling std for 7 days
daily_sales["rolling_mean_7"] = daily_sales["revenue"].rolling(window=7).mean()
daily_sales["rolling_std_7"] = daily_sales["revenue"].rolling(window=7).std()
daily_sales["rolling_max_7"] = daily_sales["revenue"].rolling(window=7).max()

In [210]:
# Retrieve year, month, day_of_week, quarter, week_of_year, season from df into daily_sales
daily_sales = daily_sales.merge(
    df[['date', 'year', 'month', 'day_of_week', 'quarter', 'week_of_year', 'season']].drop_duplicates(),
    on='date',
    how='left'
)

## Data clean

In [211]:
daily_sales.drop(columns=['date'], inplace=True)
daily_sales.dropna(inplace=True)
daily_sales.sample(10)

,revenue,quantity,avg_discount,avg_price,customers,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,year,month,day_of_week,quarter,week_of_year,season
377,"45,659,913,481.00",103,0.10,"442,851,734.50",37,"28,954,414,952.40","34,329,409,404.50","91,771,296,446.20","40,761,542,490.90","31,046,017,497.41","96,073,441,417.10",2023,4,5,2,15,Spring
403,"20,962,996,097.40",45,0.11,"521,359,195.00",20,"36,394,455,419.00","33,984,778,572.50","38,587,737,211.80","30,129,748,991.80","17,572,434,364.48","56,772,452,336.60",2023,5,1,2,20,Spring
149,"14,270,782,660.40",37,0.10,"453,349,799.00",20,"36,304,954,089.60","80,733,978,458.00","34,806,380,073.50","26,963,672,366.70","17,872,921,657.62","58,430,304,235.80",2022,7,6,3,28,Summer
253,"48,939,090,743.20",103,0.11,"538,844,395.60",50,"61,973,133,583.20","46,458,126,542.80","25,039,417,480.00","70,259,677,128.90","31,315,690,568.17","115,877,779,151.40",2022,11,2,4,47,Autumn
568,"27,280,598,907.10",59,0.12,"480,842,476.80",24,"79,783,881,583.00","8,837,952,517.10","64,970,623,605.30","72,631,948,884.70","53,720,596,286.77","173,628,565,646.90",2023,11,2,4,46,Autumn
203,"112,803,948,744.20",246,0.11,"501,012,933.33",96,"100,942,601,864.30","123,176,004,448.00","83,485,687,660.90","75,492,034,139.50","32,591,007,555.00","112,803,948,744.20",2022,9,6,3,38,Autumn
337,"89,188,581,705.30",193,0.11,"491,611,770.53",75,"24,151,312,166.10","47,750,270,346.10","16,825,397,920.80","25,829,164,499.20","29,164,247,897.12","89,188,581,705.30",2023,3,4,1,9,Spring
368,"12,121,031,090.70",39,0.20,"426,127,930.67",15,"29,761,063,525.20","44,587,927,160.00","20,820,559,912.60","25,755,324,338.60","33,143,425,379.97","98,248,301,270.00",2023,4,2,2,14,Spring
408,"31,326,222,309.20",71,0.12,"545,313,394.00",34,"29,024,832,309.90","46,400,797,989.70","37,207,869,131.30","32,551,089,295.00","11,987,899,425.43","47,413,413,173.00",2023,5,6,2,20,Spring
290,"36,645,173,953.50",70,0.08,"547,479,595.60",25,"2,909,387,415.20","2,856,124,782.00","73,809,531,751.20","50,356,838,901.80","43,293,103,656.94","105,780,377,497.20",2022,12,4,4,52,Winter


## Data encoding

In [212]:
from sklearn.preprocessing import LabelEncoder

df_ml = daily_sales.copy()
le = LabelEncoder()

# Encode the object data in dataframe
for col in df_ml.select_dtypes(include=['object', 'category']).columns:
    df_ml[col] = le.fit_transform(df_ml[col])

In [213]:
df_ml.head()

,revenue,quantity,avg_discount,avg_price,customers,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,year,month,day_of_week,quarter,week_of_year,season
30,"8,084,202,280.00",26,0.19,"352,605,799.00",10,"11,972,075,150.00","35,694,521,187.50","30,860,360,825.00","18,080,815,858.70","14,198,589,999.91","40,664,374,902.90",2022,2,0,1,8,3
31,"5,499,543,000.00",12,0.12,"581,077,000.00",5,"8,084,202,280.00","5,598,360,990.60","21,199,481,843.40","18,066,699,002.90","14,213,110,732.26","40,664,374,902.90",2022,2,1,1,8,3
32,"11,382,361,151.00",22,0.12,"649,620,699.00",10,"5,499,543,000.00","40,664,374,902.90","12,338,225,620.00","13,883,554,181.20","10,194,848,473.35","36,283,226,127.40",2022,2,2,1,8,3
33,"6,308,733,200.00",20,0.15,"430,860,500.00",10,"11,382,361,151.00","36,283,226,127.40","35,329,496,351.80","9,601,483,763.00","2,912,311,009.65","12,612,860,960.00",2022,2,4,1,8,3
34,"24,148,336,800.00",62,0.11,"447,951,000.00",25,"6,308,733,200.00","11,350,610,600.00","11,695,944,481.20","11,429,730,363.00","6,272,206,173.23","24,148,336,800.00",2022,2,6,1,8,3


## Normalize data skewed

In [214]:
skewed_features = ["revenue", "avg_price", "lag_1", "lag_7", "lag_30", "rolling_mean_7", "rolling_std_7", "rolling_max_7"]
for feature in skewed_features:
    df_ml[feature] = np.log1p(df_ml[feature])
df_ml.head(2)

,revenue,quantity,avg_discount,avg_price,customers,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,year,month,day_of_week,quarter,week_of_year,season
30,22.81,26,0.19,19.68,10,23.21,24.30,24.15,23.62,23.38,24.43,2022,2,0,1,8,3
31,22.43,12,0.12,20.18,5,22.81,22.45,23.78,23.62,23.38,24.43,2022,2,1,1,8,3


## Split data into X and y

In [215]:
from sklearn.model_selection import train_test_split
X = df_ml[["year", "month", "day_of_week", "quarter", "week_of_year", "season", "lag_1", "lag_7", "lag_30",
    "rolling_mean_7", "rolling_std_7", "rolling_max_7", "quantity"]]
y = df_ml["revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (465, 13), y_train shape: (465,)


In [216]:
X_train.head(2)

,year,month,day_of_week,quarter,week_of_year,season,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,quantity
463,2023,7,5,3,29,2,24.43,23.90,25.03,24.67,24.01,25.25,134
239,2022,11,1,4,45,0,24.75,25.18,24.16,24.73,24.48,25.70,313


## Data scaling

In [217]:
from sklearn.preprocessing import StandardScaler
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = pd.DataFrame(feature_scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(feature_scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

### Machine learning models - Demand prediction (Quantity as a target)

In [218]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0, objective='reg:squarederror', tree_method='hist')
}

parameters = {
    "Decision Tree": {
        "max_depth": [None, 5, 10, 15],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    "CatBoost": {
        "iterations": [200, 300],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7]
    },
    "XGBoost": {
        "n_estimators": [200, 300],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.8, 0.9, 1.0]
    }
}

# Train and evaluate models
trained_models = {}
results = []
feature_importances = {}

for model_name, model in models.items():
    print("=" * 50)
    print(f"Training {model_name}...")
    print("=" * 50)

    # Machine learning models with hyperparameter tuning using RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=parameters[model_name],
        n_iter=10,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    random_search.fit(X_train_scaled, y_train_scaled) # Train the model

    # Get the best model from RandomizedSearchCV
    best_model = random_search.best_estimator_
    print(f"Best parameters for {model_name}: {random_search.best_params_}")
    
    # Store the trained model
    trained_models[model_name] = best_model

    # Best parameters
    print("\nBest Parameters")
    print(random_search.best_params_)

    # Predict on the test set
    y_pred = best_model.predict(X_test_scaled)

    # Evaluate the model
    mse = mean_squared_error(y_test_scaled, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_scaled, y_pred)
    mape = mean_absolute_percentage_error(y_test_scaled, y_pred)
    r2 = r2_score(y_test_scaled, y_pred)

    # Append results to the list
    results.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

    print(f"{model_name} Evaluation Metrics:")
    print(f"Mean Squared Error (MSE): {mse:.3f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Absolute Error (MAE): {mae:.3f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.3f}")
    print(f"R-squared (R2): {r2:.3f}")

    # Feature importance
    if hasattr(best_model, 'feature_importances_'):
        importance = pd.DataFrame({
            "Feature": X_train.columns,
            "Importance": best_model.feature_importances_
        }).sort_values(by="Importance", ascending=False)
        feature_importances[model_name] = importance

# ==========================================================
# Comparison Table
# ==========================================================
df_comparison = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("\nModel Comparison:")
print(df_comparison)

# Feature importance for each model
for model_name, importance in feature_importances.items():
    print("\n")
    print("=" * 50)
    print(f"{model_name} Feature Importance:")
    print("=" * 50)
    print(importance.head(20))

Training Decision Tree...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for Decision Tree: {'min_samples_split': 2, 'min_samples_leaf': 8, 'max_depth': 10}

Best Parameters
{'min_samples_split': 2, 'min_samples_leaf': 8, 'max_depth': 10}
Decision Tree Evaluation Metrics:
Mean Squared Error (MSE): 0.040
Root Mean Squared Error (RMSE): 0.199
Mean Absolute Error (MAE): 0.139
Mean Absolute Percentage Error (MAPE): 0.966
R-squared (R2): 0.963
Training Random Forest...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for Random Forest: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}

Best Parameters
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}
Random Forest Evaluation Metrics:
Mean Squared Error (MSE): 0.117
Root Mean Squared Error (RMSE): 0.341
Mean Absolute Error (MAE): 0.244
Mean Absolute Percentage Error 

In [219]:
importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": trained_models["Random Forest"].feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print(importance)

           Feature  Importance
12        quantity        0.57
9   rolling_mean_7        0.10
11   rolling_max_7        0.05
10   rolling_std_7        0.05
7            lag_7        0.04
4     week_of_year        0.04
6            lag_1        0.04
8           lag_30        0.03
1            month        0.03
2      day_of_week        0.02
3          quarter        0.01
5           season        0.01
0             year        0.00


## Implement models prediction to database

In [220]:
df_sales_prediction = df_ml.copy()

# Store model performancy summary
model_summary = []

for model_name, model in trained_models.items():
    print("=" * 70)
    print(f"Predicting Revenue using {model_name}")
    print("=" * 70)

    # Scale ALL feature using training scaler
    X_all_scaled = pd.DataFrame(feature_scaler.transform(X), columns=X.columns, index=X.index)

    # Predict on scaled data
    y_pred_scaled = model.predict(X_all_scaled)

    # Convert back to original revenue scale
    y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Save prediction
    df_sales_prediction[f"{model_name}_predicted_revenue"] = np.round(y_pred, 2)

    # Row-level error analysis
    df_sales_prediction[f"{model_name}_absolute_error"] = (df_sales_prediction["revenue"] - y_pred).abs()
    df_sales_prediction[f"{model_name}_squared_error"] = (df_sales_prediction["revenue"] - y_pred) ** 2

    # Overall model performance metrics
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_pred)
    mape = mean_absolute_percentage_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    model_summary.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

    print(f"{model_name} Overall Performance Metrics:")
    print(f"Mean Squared Error (MSE): {mse:.3f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Absolute Error (MAE): {mae:.3f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.3f}")
    print(f"R-squared (R2): {r2:.3f}")

Predicting Revenue using Decision Tree
Decision Tree Overall Performance Metrics:
Mean Squared Error (MSE): 0.018
Root Mean Squared Error (RMSE): 0.133
Mean Absolute Error (MAE): 0.093
Mean Absolute Percentage Error (MAPE): 0.004
R-squared (R2): 0.978
Predicting Revenue using Random Forest
Random Forest Overall Performance Metrics:
Mean Squared Error (MSE): 0.029
Root Mean Squared Error (RMSE): 0.170
Mean Absolute Error (MAE): 0.109
Mean Absolute Percentage Error (MAPE): 0.005
R-squared (R2): 0.965
Predicting Revenue using CatBoost
CatBoost Overall Performance Metrics:
Mean Squared Error (MSE): 0.017
Root Mean Squared Error (RMSE): 0.130
Mean Absolute Error (MAE): 0.096
Mean Absolute Percentage Error (MAPE): 0.004
R-squared (R2): 0.980
Predicting Revenue using XGBoost
XGBoost Overall Performance Metrics:
Mean Squared Error (MSE): 0.007
Root Mean Squared Error (RMSE): 0.084
Mean Absolute Error (MAE): 0.051
Mean Absolute Percentage Error (MAPE): 0.002
R-squared (R2): 0.992


In [221]:
df_sales_prediction

,revenue,quantity,avg_discount,avg_price,customers,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,year,month,day_of_week,quarter,week_of_year,season,Decision Tree_predicted_revenue,Decision Tree_absolute_error,Decision Tree_squared_error,Random Forest_predicted_revenue,Random Forest_absolute_error,Random Forest_squared_error,CatBoost_predicted_revenue,CatBoost_absolute_error,CatBoost_squared_error,XGBoost_predicted_revenue,XGBoost_absolute_error,XGBoost_squared_error
30,22.81,26,0.19,19.68,10,23.21,24.30,24.15,23.62,23.38,24.43,2022,2,0,1,8,3,23.12,0.31,0.09,22.95,0.14,0.02,23.16,0.34,0.12,22.93,0.12,0.01
31,22.43,12,0.12,20.18,5,22.81,22.45,23.78,23.62,23.38,24.43,2022,2,1,1,8,3,22.20,0.22,0.05,22.56,0.13,0.02,22.27,0.15,0.02,22.37,0.06,0.00
32,23.16,22,0.12,20.29,10,22.43,24.43,23.24,23.35,23.05,24.31,2022,2,2,1,8,3,22.93,0.23,0.05,23.01,0.15,0.02,22.97,0.19,0.04,22.89,0.27,0.07
33,22.57,20,0.15,19.88,10,23.16,24.31,24.29,22.99,21.79,23.26,2022,2,4,1,8,3,22.93,0.36,0.13,22.69,0.13,0.02,22.82,0.25,0.06,22.60,0.03,0.00
34,23.91,62,0.11,19.92,25,22.57,23.15,23.18,23.16,22.56,23.91,2022,2,6,1,8,3,23.95,0.04,0.00,23.83,0.08,0.01,23.96,0.05,0.00,23.94,0.04,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607,24.80,137,0.11,19.99,67,25.19,25.30,25.59,24.73,24.22,25.43,2023,12,1,4,52,3,24.70,0.10,0.01,24.82,0.02,0.00,24.81,0.01,0.00,24.75,0.06,0.00
608,24.15,63,0.10,20.02,34,24.80,24.15,25.00,24.73,24.22,25.43,2023,12,2,4,52,3,24.16,0.01,0.00,24.18,0.03,0.00,24.00,0.15,0.02,24.12,0.03,0.00
609,25.76,333,0.12,20.08,133,24.15,23.68,25.15,25.03,24.55,25.76,2023,12,4,4,52,3,25.68,0.08,0.01,25.49,0.27,0.07,25.67,0.10,0.01,25.76,0.00,0.00
610,24.45,93,0.12,20.09,40,25.76,24.40,25.05,25.03,24.54,25.76,2023,12,5,4,52,3,24.43,0.01,0.00,24.49,0.05,0.00,24.41,0.03,0.00,24.44,0.00,0.00


In [222]:
# Retrieve sales actual data from original df to compare with predicted revenue
df_sales_prediction = df_sales_prediction.merge(
    df[['date', 'sales']].drop_duplicates(),
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
# Model performance summary
df_model_summary = pd.DataFrame(model_summary).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("\nModel Performance Summary:")
print(df_model_summary)

# Prediction columns to display
prediction_columns = [
    "sales",
    "Decision Tree_predicted_revenue",
    "Random Forest_predicted_revenue",
    "CatBoost_predicted_revenue",
    "XGBoost_predicted_revenue"
]

# Inverse value in prediction_columns to original scale


print("\nPrediction Preview")
display(df_sales_prediction[prediction_columns].head(20))


Model Performance Summary:
           Model  MSE  RMSE  MAE  MAPE   R2
0        XGBoost 0.01  0.08 0.05  0.00 0.99
1       CatBoost 0.02  0.13 0.10  0.00 0.98
2  Decision Tree 0.02  0.13 0.09  0.00 0.98
3  Random Forest 0.03  0.17 0.11  0.00 0.96

Prediction Preview


,sales,Decision Tree_predicted_revenue,Random Forest_predicted_revenue,CatBoost_predicted_revenue,XGBoost_predicted_revenue
30,"352,604,000.00",44.92,44.77,44.96,44.75
31,"528,906,000.00",44.09,44.42,44.15,44.24
32,NaN,44.75,44.82,44.79,44.71
33,"1,516,197,200.00",44.75,44.53,44.65,44.45
34,"287,840,000.00",45.67,45.56,45.68,45.66
35,"377,790,000.00",44.75,44.69,44.69,44.66
36,"717,801,000.00",45.91,45.68,45.96,45.96
37,"823,042,500.00",45.33,45.35,45.32,45.42
38,"3,437,889,000.00",46.76,46.59,46.73,46.79
39,"282,083,200.00",45.15,45.10,45.19,45.16


## Save models